# StoryZop: Instagram Story Analyzer
This notebook runs the complete StoryZop pipeline on Google Colab, leveraging Playwright for browser automation and Qwen3-VL models for vision analysis.

## Section 1: Install Dependencies
Install the required libraries for browser automation, AI models, and OCR.

In [ ]:
!pip install playwright Pillow pydantic pydantic-settings python-dotenv nest-asyncio
!playwright install chromium
!pip install torch transformers accelerate qwen-vl-utils bitsandbytes easyocr

import nest_asyncio
nest_asyncio.apply()

## Section 2: GPU/Environment Check
Check the current GPU allocation to ensure the models will fit in VRAM.

In [ ]:
import sys
import os
# Ensure the src directory is available if you uploaded the project folder
sys.path.append(os.path.abspath('..'))

from src.vision.gpu import GPUManager
GPUManager.print_gpu_status()

## Section 3: Configuration
Set the core configuration values for the pipeline.

In [ ]:
from src.config import get_config

# You can override defaults here. For Colab, we might want to use 4-bit quantization if on a T4 GPU.
config = get_config(
    use_4bit_quantization=True, 
    headless=True
)
print("Configured Data Directory:", config.data_dir)
print("Configured Models:", config.initial_model, "|", config.primary_model)

## Section 4: Secure Authentication Setup
Load Instagram cookies to authenticate the browser session securely.
Upload your `cookies.json` to the Colab environment or load from Colab Secrets.

In [ ]:
import json

cookies_path = 'cookies.json'
if not os.path.exists(cookies_path):
    print("⚠️ Please upload 'cookies.json' to authenticate.")
else:
    with open(cookies_path, 'r') as f:
        cookies = json.load(f)
    print(f"Loaded {len(cookies)} cookies.")

## Section 5: Load Models
Initialize the Qwen3-VL models. Loading 32B might require high-RAM and A100.

In [ ]:
from src.vision.qwen4b import Qwen4BScreener
from src.vision.qwen8b import Qwen8BAnalyzer
from src.vision.qwen32b import Qwen32BExpert
from src.vision.ocr import OCREngine

print("Initializing models (lazy loading)...")
screener = Qwen4BScreener(config)
analyzer = Qwen8BAnalyzer(config)
expert = Qwen32BExpert(config)
ocr_engine = OCREngine()

# Optional: load them now into VRAM, or let them load lazily when used
# screener.load_model()

## Section 6: Initialize Database
Initialize the local SQLite database.

In [ ]:
from src.database.database import Database

db = Database(config.db_path)
db.initialize()

state = db.get_processing_state() if hasattr(db, 'get_processing_state') else {}
print("Database initialized. State:", state)

## Section 7: Initialize Browser
Launch the Playwright browser and load the cookies.

In [ ]:
from src.browser.session import BrowserSession
import asyncio

session = BrowserSession(config)
await session.launch()

if os.path.exists(cookies_path):
    await session.load_cookies(cookies_path)

# await session.page.goto('https://instagram.com/')

## Section 8: Scan Stories
Find available stories from the Instagram tray.

In [ ]:
# In a full implementation, you'd have a Navigation or Sampler class here to extract the tray.
print("Scanning stories tray... (Mock)")
# discovered_items = await navigator.get_story_tray_items()
# print(f"Found {len(discovered_items)} stories.")

## Section 9: Initial Analysis
First pass: capture frames -> OCR -> Qwen-4B screening.

In [ ]:
from src.pipeline import StoryPipeline

pipeline = StoryPipeline(config, db)
pipeline.set_browser(session)
pipeline.set_models(screener, analyzer, expert)
pipeline.set_ocr(ocr_engine)

# If we had the navigator and sampler classes initialized, we could inject them here:
# pipeline.set_components(navigator, sampler)

print("Ready to run initial analysis pass.")

## Section 10: Revisit Queue
Display stories that need more frames.

In [ ]:
revisit_queue = db.get_revisit_queue()
print(f"{len(revisit_queue)} stories in revisit queue.")
for r in revisit_queue:
    print(f"Revisit Story {r.story_id}: priority {r.priority}")

## Section 11: Revisit Processing
Revisit the queued stories and capture the extended frames.

In [ ]:
print("Processing revisit queue...")
# For a full run, this is handled via pipeline.run()
# await pipeline.run()

## Section 12: Final Analysis
Run the 8B detailed analysis, followed by the 32B expert review if confidence is low.

In [ ]:
print("Final analysis pending...")
# This is also managed within pipeline.run()
pending = db.get_pending_stories()
print(f"{len(pending)} stories waiting for final analysis.")

## Section 13: Export Results
Generate text reports, JSON, and CSV data.

In [ ]:
from src.analysis.report import ReportGenerator

report_gen = ReportGenerator(db)

print("--- Text Report ---")
print(report_gen.generate_text_report())

json_path = config.data_dir / "export.json"
csv_path = config.data_dir / "export.csv"

report_gen.export_json(json_path)
report_gen.export_csv(csv_path)

print(f"Exported JSON to {json_path}")
print(f"Exported CSV to {csv_path}")

# Cleanup browser
await session.close()